# S3.15 — Aggregations: groupBy + agg
**Date completed:** October 2026  
**Status:** Completed  
**Dataset:** samples.tpch.orders — 7.5M rows
**Key concept:** groupBy = divide into groups, agg = summarise each group

In [0]:
# ============================================================
# S3.15 — Cell 2
# Aggregations: groupBy() + agg()
#
# Goal:
# 1. Understand how aggregation changes DataFrame grain.
# 2. Perform global aggregation using agg().
# 3. Perform grouped aggregation using groupBy() + agg().
# 4. Apply multiple aggregation functions together.
# 5. Use count, sum, avg, min, max, and stddev.
# 6. Understand count("*"), count(column), and countDistinct().
# 7. Group by multiple columns and derived date expressions.
# 8. Understand NULL behaviour in aggregate functions.
# 9. Filter aggregated results (SQL HAVING pattern).
# 10. Build Gold-style daily revenue and customer summaries.
# 11. Understand shuffle/performance implications.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# Important:
# Aggregation changes the grain of the DataFrame.
#
# Example:
# Order-level rows
#      ↓
# groupBy(status)
#      ↓
# One row per status
# ============================================================

import pyspark.sql.functions as F


print("=== S5: AGGREGATIONS — groupBy() + agg() ===")
print()


# ------------------------------------------------------------
# Step 1 — Load Source Data
# ------------------------------------------------------------

df_orders = spark.table(
    "samples.tpch.orders"
)

print("Orders DataFrame loaded.")

source_count = df_orders.count()

print(
    f"Source order rows: {source_count:,}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 2 — Understand Aggregation Grain
# Before aggregation:
# Grain:
# One row per order.
# After:
# groupBy("o_orderstatus")
# Grain becomes:
# One row per distinct order status.
# groupBy() therefore changes the level of detail of the
# resulting DataFrame.
# ------------------------------------------------------------

print("=== STEP 2: AGGREGATION GRAIN ===")

print("""
Source grain:
→ One row per order

groupBy("o_orderstatus"):
→ One row per order status

groupBy("year", "month", "status"):
→ One row per year + month + status combination

Always understand the target grain before aggregating.
""")

In [0]:
# ------------------------------------------------------------
# Step 3 — Global Aggregation
# groupBy() is NOT required when we want one summary row
# across the entire DataFrame.
# DataFrame.agg(...)
# performs a global aggregation.
# ------------------------------------------------------------

print("=== STEP 3: GLOBAL AGGREGATION ===")

df_global = df_orders.agg(

    F.count("*").alias(
        "total_orders"
    ),

    F.sum(
        "o_totalprice"
    ).alias(
        "total_revenue"
    ),

    F.avg(
        "o_totalprice"
    ).alias(
        "avg_order_value"
    ),

    F.min(
        "o_totalprice"
    ).alias(
        "min_order_value"
    ),

    F.max(
        "o_totalprice"
    ).alias(
        "max_order_value"
    )
)

df_global.show(
    truncate=False
)
print()

In [0]:
# ------------------------------------------------------------
# Step 4 — Basic groupBy() + agg()
# Business question:
# What is the total revenue for each order status?
# groupBy():
#     defines the grouping key.
# agg():
#     defines the calculations performed for each group.
# ------------------------------------------------------------

print("=== STEP 4: BASIC groupBy() + agg() ===")

df_basic = (

    df_orders.groupBy("o_orderstatus")

    .agg(F.sum("o_totalprice").alias("total_revenue")))

df_basic.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 5 — Multiple Aggregations Together
# Related measures can be calculated within the same grouped
# aggregation.
# This commonly allows Spark to share grouping work rather
# than independently repeating equivalent groupBy operations.
# Do NOT use the rule:
# "one agg always equals exactly one shuffle"
# Physical execution depends on the query and optimizer.
# ------------------------------------------------------------

print(
    "=== STEP 5: MULTIPLE AGGREGATIONS ==="
)
df_multi = (df_orders.groupBy("o_orderstatus")
            .agg(F.count("*").alias("order_count"),
                 
                 F.sum("o_totalprice").alias("total_revenue"),

                 F.round(F.avg("o_totalprice"),2).alias("avg_order_value"),

                 F.min("o_totalprice").alias("min_order_value"),

                 F.max("o_totalprice").alias("max_order_value"),
                 
                 F.round(F.stddev("o_totalprice"),2).alias("stddev_order_value"))
            
            .orderBy(F.col("order_count").desc()))

df_multi.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 6 — Aggregate Function Meaning
# ------------------------------------------------------------

print("=== STEP 6: AGGREGATE FUNCTIONS ===")

print("""
count()
→ Count records / non-NULL values depending on expression.

sum()
→ Add numeric values.

avg()
→ Arithmetic mean of non-NULL numeric values.

min()
→ Minimum non-NULL value.

max()
→ Maximum non-NULL value.

stddev()
→ Sample standard deviation of non-NULL numeric values.
""")

In [0]:
# ------------------------------------------------------------
# Step 7 — count("*") vs count(column) vs countDistinct()
#
# count("*"):
# → Counts rows.
#
# count(column):
# → Counts NON-NULL values in that column.
#
# countDistinct(column):
# → Counts distinct NON-NULL values.
# ------------------------------------------------------------

print(
    "=== STEP 7: COUNT VARIANTS ==="
)


df_counts = df_orders.agg(

    F.count("*").alias("total_rows"),

    F.count("o_orderstatus").alias("non_null_status_count"),

    F.countDistinct("o_orderstatus").alias("distinct_status_count"),

    F.countDistinct("o_custkey").alias("distinct_customer_count"))


df_counts.show(truncate=False)

print("""count("*")
→ all rows

count("column")
→ non-NULL values in that column

countDistinct("column")
→ distinct non-NULL values

These operations answer different questions.
""")

In [0]:
# ------------------------------------------------------------
# Step 8 — Aggregation and NULL Values
#
# Most numeric aggregations such as:
#
# sum()
# avg()
# min()
# max()
# stddev()
#
# ignore NULL input values.
#
# count(column) also ignores NULL.
#
# count("*") counts the row regardless of whether individual
# columns contain NULL.
# ------------------------------------------------------------

print(
    "=== STEP 8: NULL BEHAVIOUR IN AGGREGATION ==="
)


null_agg_data = [

    ("A", 100.0),

    ("A", 200.0),

    ("A", None),

    ("B", 50.0),

    ("B", None)
]


df_null_agg = spark.createDataFrame(
    null_agg_data,["group_id","amount"])


df_null_agg_result = (

    df_null_agg

    .groupBy("group_id")

    .agg(F.count("*").alias("row_count"),

        F.count("amount").alias("non_null_amount_count"),

        F.sum("amount").alias("amount_sum"),

        F.avg("amount").alias("amount_avg"))

    .orderBy("group_id"))


df_null_agg_result.show(truncate=False)


print("""
For group A:

Rows:
100
200
NULL

count("*")
→ 3

count("amount")
→ 2

sum("amount")
→ 300

avg("amount")
→ 150

NULL does not automatically behave like zero.
""")

In [0]:
# ------------------------------------------------------------
# Step 9 — groupBy Multiple Columns
#
# Business question:
#
# How many orders and how much revenue exist for each:
#
# year
# month
# status
#
# Target grain:
#
# one row per
# year + month + status
# ------------------------------------------------------------

print(
    "=== STEP 9: MULTI-COLUMN GROUPING ==="
)

df_monthly = (
    df_orders

    .groupBy(

        F.year("o_orderdate").alias("order_year"),

        F.month("o_orderdate").alias("order_month"),

        F.col("o_orderstatus").alias("status"))

    .agg(F.count("*").alias("order_count"),

        F.sum("o_totalprice").alias("monthly_revenue"))

    .orderBy("order_year","order_month","status"))


print("\nMonthly revenue by year/month/status:")

df_monthly.show(15,truncate=False)


monthly_summary_count = (df_monthly.count())


print(
    f"Total monthly summary rows: "
    f"{monthly_summary_count:,}"
)

print()

In [0]:
# ------------------------------------------------------------
# Step 10 — groupBy().count() Shorthand
# If the only measure required is row count:
# groupBy(...).count()
# is a convenient shorthand.
# ------------------------------------------------------------

print(
    "=== STEP 10: groupBy().count() ==="
)

df_status_count = (

    df_orders

    .groupBy("o_orderstatus")

    .count()

    .orderBy("o_orderstatus"))


df_status_count.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 11 — Filtering Aggregated Results
#
# SQL concept:
#
# HAVING
#
# DataFrame pattern:
#
# groupBy()
#     ↓
# agg()
#     ↓
# filter()
#
# Example requirement:
#
# Show customer groups whose total revenue exceeds
# 1,000,000.
# ------------------------------------------------------------

print(
    "=== STEP 11: FILTER AGGREGATED RESULTS ==="
)


df_high_revenue_customers = (

    df_orders

    .groupBy("o_custkey")

    .agg(
        F.count("*").alias("order_count"),

        F.sum("o_totalprice").alias("total_revenue"))

    .filter(F.col("total_revenue") > 1000000)

    .orderBy(F.col("total_revenue").desc()))


df_high_revenue_customers.show(10,truncate=False)


print("""
SQL idea:

GROUP BY
    ↓
HAVING

PySpark DataFrame idea:

groupBy()
    ↓
agg()
    ↓
filter()
""")


In [0]:
# ------------------------------------------------------------
# Step 12 — Gold Pattern: Daily Revenue Summary
#
# This now matches our stated learning goal.
#
# Target grain:
#
# ONE ROW PER ORDER DATE
#
# Measures:
#
# order_count
# unique_customers
# total_revenue
# average_order_value
# min_order_value
# max_order_value
# ------------------------------------------------------------

print(
    "=== STEP 12: GOLD — DAILY REVENUE SUMMARY ==="
)


df_daily_revenue = (

    df_orders

    .groupBy(

        F.col("o_orderdate").alias("order_date"))

    .agg(
        F.count("*").alias("order_count"),

        F.countDistinct("o_custkey").alias("unique_customers"),

        F.sum("o_totalprice").alias("total_revenue"),

        F.round(F.avg("o_totalprice"),2).alias("avg_order_value"),

        F.min("o_totalprice").alias("min_order_value"),

        F.max("o_totalprice").alias("max_order_value"))

    .orderBy("order_date"))


df_daily_revenue.show(10,truncate=False)

print()


In [0]:
# ------------------------------------------------------------
# Step 13 — Gold Pattern: Top 10 Customers by Revenue
#
# Target grain:
#
# ONE ROW PER CUSTOMER
#
# Then:
#
# orderBy(total_revenue DESC)
#        ↓
# limit(10)
# ------------------------------------------------------------

print(
    "=== STEP 13: GOLD — TOP 10 CUSTOMERS ==="
)


df_top_customers = (

    df_orders

    .groupBy("o_custkey")

    .agg(

        F.count("*").alias("order_count"),

        F.sum("o_totalprice").alias("total_revenue"),

        F.round(F.avg("o_totalprice"),2).alias("avg_order_value"))
    .orderBy(F.col("total_revenue").desc())
    .limit(10))


df_top_customers.show(truncate=False)

print()

In [0]:
# ------------------------------------------------------------
# Step 14 — Aggregation Reconciliation
#
# Important validation:
#
# If every source row belongs to exactly one status group:
#
# SUM(group order_count)
#
# should equal:
#
# source row count
# ------------------------------------------------------------

print("=== STEP 14: AGGREGATION VALIDATION ===")

grouped_order_count = (

    df_multi

    .agg(
        F.sum("order_count").alias("reconciled_count"))

    .first()["reconciled_count"])


print(
    f"Source rows             : "
    f"{source_count:,}")

print(
    f"Grouped/reconciled rows : "
    f"{grouped_order_count:,}"
)


assert grouped_order_count == source_count, ("Grouped order counts do not reconcile to source")


print("Aggregation count reconciliation passed.")

print()


In [0]:
# ------------------------------------------------------------
# Step 15 — Revenue Reconciliation
#
# Total revenue across all status groups should match the
# global total revenue, subject to the same NULL semantics.
# ------------------------------------------------------------

print(
    "=== STEP 15: REVENUE RECONCILIATION ==="
)


global_revenue = (

    df_orders

    .agg(
        F.sum("o_totalprice").alias("total_revenue"))

    .first()["total_revenue"])

grouped_revenue = (

    df_basic

    .agg(
        F.sum("total_revenue").alias("total_revenue"))

    .first()["total_revenue"])

print(f"Global revenue  : {global_revenue}")

print(f"Grouped revenue : {grouped_revenue}")

assert grouped_revenue == global_revenue, ("Grouped revenue does not reconcile to global revenue")

print("Revenue reconciliation passed.")

print()

In [0]:
# ------------------------------------------------------------
# Step 16 — Performance Awareness
#
# groupBy() commonly requires records with the same grouping
# key to be brought together.
#
# This often involves:
#
# partial aggregation
#       ↓
# shuffle / Exchange
#       ↓
# final aggregation
#
# Spark may perform partial aggregation before the shuffle
# to reduce the amount of data transferred.
#
# Exact physical execution is determined by Spark's optimizer.
# ------------------------------------------------------------

print(
    "=== STEP 16: PERFORMANCE AWARENESS ==="
)


print("""
groupBy() is commonly a wide transformation.

Conceptually:

Input partitions
      ↓
Partial aggregation where possible
      ↓
Exchange / shuffle by grouping key
      ↓
Final aggregation
      ↓
Grouped result


Important:

Do NOT use the absolute rule:

"Every groupBy always causes exactly one shuffle."

Physical execution depends on:

- aggregation functions
- distinct aggregates
- optimizer decisions
- Adaptive Query Execution
- data distribution
- partitioning
""")

In [0]:
# ------------------------------------------------------------
# Step 17 — countDistinct() Performance Note
#
# Distinct aggregation requires Spark to identify unique values
# across the grouping scope.
#
# It may require additional aggregation / exchange work compared
# with a simple count().
#
# Therefore use it because the metric requires distinct values,
# not as a default replacement for count().
# ------------------------------------------------------------

print(
    "=== STEP 17: countDistinct() AWARENESS ==="
)


print("""
count("*")
→ count rows

count(column)
→ count non-NULL values

countDistinct(column)
→ count unique non-NULL values

countDistinct() can require more work than a normal count()
because Spark must determine uniqueness across the data.
""")

In [0]:
# ------------------------------------------------------------
# Step 18 — Transformations vs Actions
# ------------------------------------------------------------

print(
    "=== STEP 18: TRANSFORMATIONS VS ACTIONS ==="
)


print("""
Transformations:

groupBy()
agg()
orderBy()
filter()
limit()
select()

Aggregate Column functions:

count()
sum()
avg()
min()
max()
stddev()
countDistinct()

Actions used:

show()
count()
first()

Important:

An aggregation DataFrame is still lazily evaluated.

Execution occurs when an action requests the result.
""")


In [0]:
# ------------------------------------------------------------
# Step 19 — Final Learning Summary
# ------------------------------------------------------------

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
GLOBAL AGGREGATION:

df.agg(...)

→ One summary across the entire DataFrame.


GROUPED AGGREGATION:

df.groupBy(keys).agg(...)

→ One summary row per grouping-key combination.


GRAIN:

Before groupBy:
→ detailed source grain

After groupBy:
→ grouping-key grain


AGGREGATION FUNCTIONS:

count()
sum()
avg()
min()
max()
stddev()
countDistinct()


COUNT DIFFERENCES:

count("*")
→ all rows

count(column)
→ non-NULL values

countDistinct(column)
→ unique non-NULL values


NULL BEHAVIOUR:

Most numeric aggregate functions ignore NULL values.

NULL is not automatically treated as zero.


MULTIPLE AGGREGATIONS:

Related measures can be calculated within the same agg()
to allow Spark to optimise shared grouping work.


MULTIPLE GROUPING KEYS:

groupBy(
    year,
    month,
    status
)

→ one row per unique year/month/status combination.


HAVING PATTERN:

groupBy()
    ↓
agg()
    ↓
filter()

GOLD PATTERN:

Detailed Silver data
       ↓
groupBy(business dimensions)
       ↓
agg(business measures)
       ↓
Validate grain and totals
       ↓
Gold summary


PERFORMANCE:

groupBy() commonly requires a shuffle.

Spark may use partial aggregation before the shuffle.

countDistinct() can require additional work.

Do not assume the physical plan without inspecting it.


VALIDATION:

Always reconcile important aggregates when practical.

Example:

SUM(group counts)
→ should match source count when every source row belongs
  to exactly one group.


CORE PRINCIPLE:

Before writing an aggregation, ask:

"What should ONE output row represent?"

That answer defines the aggregation grain.
""")


print("=== S3.15 CELL 2 COMPLETED ===")

# S3.15 — Key Takeaways: Aggregations

## 1. Two Forms

| Form | Syntax | Output Grain |
|---|---|---|
| Global aggregation | `df.agg(...)` | One summary row for the whole DataFrame |
| Grouped aggregation | `df.groupBy(keys).agg(...)` | One row per unique grouping-key combination |

---

## 2. Aggregation Functions

Common aggregation functions:

```text
count()
sum()
avg()
min()
max()
stddev()
countDistinct()
```

`stddev()` calculates the sample standard deviation.

Use `alias()` to give aggregate outputs meaningful names.

Example:

```python
F.sum(
    "o_totalprice"
).alias(
    "total_revenue"
)
```

---

## 3. count() Differences

```text
count("*")
→ Counts all rows

count(column)
→ Counts non-NULL values in that column

countDistinct(column)
→ Counts distinct non-NULL values
```

These functions answer different business questions.

---

## 4. NULL Behaviour

Most common aggregate functions ignore NULL input values.

Examples:

```text
sum()
avg()
min()
max()
stddev()
count(column)
```

Important:

```text
NULL
≠
0
```

NULL is not automatically treated as zero.

`count("*")` still counts the row even when individual columns contain NULL.

---

## 5. Multiple Aggregations

When several measures use the same grouping keys, calculate them together
in the same `agg()` when practical.

Example:

```python
df.groupBy(
    "status"
).agg(
    F.count("*").alias("order_count"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order_value")
)
```

This allows Spark to share grouping work where possible.

Do NOT use the rule:

```text
One groupBy = exactly one shuffle
```

The actual physical execution depends on:

```text
Aggregation functions
Distinct aggregates
Data distribution
Partitioning
Catalyst optimizer
Adaptive Query Execution
```

---

## 6. Multiple Grouping Keys

`groupBy()` can use more than one grouping column.

Example:

```python
df.groupBy(
    "order_year",
    "order_month",
    "status"
).agg(...)
```

Target grain becomes:

```text
One row per
year + month + status combination
```

---

## 7. groupBy().count()

When only row count is required:

```python
df.groupBy(
    "status"
).count()
```

is a convenient shorthand.

Equivalent business idea:

```text
For each status
→ count the rows
```

---

## 8. HAVING Pattern

SQL:

```text
GROUP BY
   ↓
HAVING
```

PySpark DataFrame pattern:

```text
groupBy()
   ↓
agg()
   ↓
filter()
```

Example:

```python
(
    df
    .groupBy("customer_id")
    .agg(
        F.sum("amount").alias("revenue")
    )
    .filter(
        F.col("revenue") > 1000000
    )
)
```

---

## 9. Grain

Aggregation changes the level of detail of the DataFrame.

Example:

```text
Before groupBy:
One row per order

groupBy(status):
One row per status

groupBy(customer_id):
One row per customer

groupBy(order_date):
One row per date
```

Always ask:

> **What should ONE output row represent?**

That answer defines the aggregation grain.

---

## 10. Performance

`groupBy()` commonly requires data with the same grouping key to be brought
together.

Typical conceptual flow:

```text
Input partitions
      ↓
Partial aggregation where applicable
      ↓
Exchange / Shuffle
      ↓
Final aggregation
      ↓
Grouped result
```

### Important

Spark can perform partial aggregation for suitable aggregate functions,
which can reduce data transferred during shuffle.

However, do not assume every aggregation has exactly the same physical plan.

`countDistinct()` can require additional aggregation/exchange work because
Spark must determine uniqueness across the required scope.

Use:

```python
df.explain()
```

and Databricks Query Profile when execution details matter.

---

## 11. Validation

Aggregation results should be reconciled when practical.

### Count Reconciliation

If:

- no source rows were filtered out, and
- every source row belongs to exactly one group,

then:

```text
SUM(group order_count)
=
source row count
```

Example:

```text
Status F count
+
Status O count
+
Status P count
=
Total source rows
```

### Measure Reconciliation

Important additive measures can also be reconciled.

Example:

```text
SUM(revenue across status groups)
=
global revenue
```

provided both calculations use the same source rows and NULL semantics.

---

## 12. Gold-Layer Pattern

```text
Silver detail data
       ↓
Define target grain
       ↓
groupBy(business dimensions)
       ↓
agg(business measures)
       ↓
Validate counts / totals
       ↓
Gold summary
```

Example:

```text
Order-level Silver data
       ↓
groupBy(order_date)
       ↓
order_count
unique_customers
total_revenue
avg_order_value
       ↓
One Gold row per day
```

---

## 13. Quick Decision Guide

```text
Need one summary for entire DataFrame?
→ agg()

Need one summary per category?
→ groupBy() + agg()

Need multiple measures for same grouping?
→ Calculate together when practical

Need unique-value count?
→ countDistinct()

Need only grouped row count?
→ groupBy().count()

Need to filter aggregated results?
→ groupBy() → agg() → filter()

Need physical-plan details?
→ explain() / Query Profile
```

---

## 14. Critical Rules

1. Aggregation changes DataFrame grain.

2. Define the target grain before writing `groupBy()`.

3. `count("*")`, `count(column)`, and `countDistinct(column)` have different semantics.

4. Most common numeric aggregates ignore NULL values.

5. NULL is not the same as zero.

6. Combine related measures with the same grouping keys when practical.

7. Do not assume one `groupBy()` always means exactly one shuffle.

8. `countDistinct()` can require additional work.

9. Use meaningful aliases for aggregate measures.

10. Reconcile important counts and measures whenever practical.

---

## 15. Final Understanding

```text
Detailed Data
     ↓
Choose grouping dimensions
     ↓
Define target grain
     ↓
Apply aggregate measures
     ↓
Validate results
     ↓
Business summary / Gold layer
```

Core question:

> **What should one output row represent?**

**S3.15 — Aggregations completed.**